In [1]:
# Open the persistent DuckDB / DuckLake catalog created by the earlier notebooks.
# Run notebooks in README order from the assign1 folder.
import duckdb
import pandas as pd
from pathlib import Path
from IPython.display import display
Path("lakehouse_data").mkdir(exist_ok=True)
con = duckdb.connect("hotel_pipeline.duckdb")
con.execute("INSTALL ducklake;")
con.execute("LOAD ducklake;")
con.execute("ATTACH 'ducklake:hotel_catalog.duckdb' AS hotel_lake (DATA_PATH 'lakehouse_data/');")
print("Connected to the hotel lakehouse.")


Connected to the hotel lakehouse.


## Task 4: Silver Layer — Data Cleaning and Validation

The Silver layer transforms raw hotel booking records into validated, analysis-ready data.

The following quality rules will be applied:

1. Cancellation status must be either 0 or 1.
2. Lead time and number of guests cannot be negative.
3. The arrival date must be a valid calendar date.
4. ADR (Average Daily Rate) cannot be negative.
5. A booking must contain at least one guest.

Records failing these rules will be placed in a quarantine table with an explanation.

Missing Country values will be labeled "Unknown". Missing Children values will be treated as zero only for records that otherwise pass validation, and this assumption will be documented.

Identical-looking booking records will be retained because the dataset does not provide a unique booking ID. The source file and original row number will be used to distinguish observations.

In [2]:
# Inspect important columns before Silver cleaning

quality_checks = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN IsCanceled NOT IN (0, 1)
             OR IsCanceled IS NULL THEN 1 ELSE 0 END)
        AS invalid_cancellation,

    SUM(CASE WHEN LeadTime < 0 OR LeadTime IS NULL
             THEN 1 ELSE 0 END)
        AS invalid_lead_time,

    SUM(CASE WHEN ADR < 0 OR ADR IS NULL
             THEN 1 ELSE 0 END)
        AS invalid_adr,

    SUM(CASE WHEN Children IS NULL
             THEN 1 ELSE 0 END)
        AS missing_children,

    SUM(CASE WHEN Country IS NULL
             THEN 1 ELSE 0 END)
        AS missing_country,

    SUM(CASE WHEN Adults + COALESCE(Children, 0) + Babies <= 0
             THEN 1 ELSE 0 END)
        AS zero_guests

FROM hotel_lake.bronze.hotel_bookings
""").df()

display(quality_checks)

,total_rows,invalid_cancellation,invalid_lead_time,invalid_adr,missing_children,missing_country,zero_guests
0,119390,0.0,0.0,1.0,4.0,488.0,180.0


In [3]:
# Create a validated staging table with rejection reasons
con.execute("""
CREATE OR REPLACE TABLE hotel_lake.silver.booking_validation AS
SELECT *,
    CASE
        WHEN IsCanceled IS NULL OR IsCanceled NOT IN (0, 1)
            THEN 'Invalid cancellation status'
        WHEN LeadTime IS NULL OR LeadTime < 0
            THEN 'Invalid lead time'
        WHEN ADR IS NULL OR ADR < 0
            THEN 'Invalid ADR'
        WHEN Adults IS NULL OR Adults < 0
          OR Children < 0 OR Babies IS NULL OR Babies < 0
            THEN 'Invalid guest count'
        WHEN Adults + COALESCE(Children, 0) + Babies <= 0
            THEN 'No guests'
        WHEN TRY_CAST(
            CAST(ArrivalDateYear AS VARCHAR) || '-' ||
            CAST(
                CASE ArrivalDateMonth
                    WHEN 'January' THEN 1
                    WHEN 'February' THEN 2
                    WHEN 'March' THEN 3
                    WHEN 'April' THEN 4
                    WHEN 'May' THEN 5
                    WHEN 'June' THEN 6
                    WHEN 'July' THEN 7
                    WHEN 'August' THEN 8
                    WHEN 'September' THEN 9
                    WHEN 'October' THEN 10
                    WHEN 'November' THEN 11
                    WHEN 'December' THEN 12
                END AS VARCHAR
            ) || '-' ||
            CAST(ArrivalDateDayOfMonth AS VARCHAR)
            AS DATE
        ) IS NULL
            THEN 'Invalid arrival date'
        ELSE NULL
    END AS rejection_reason
FROM hotel_lake.bronze.hotel_bookings
""")

print("Validation staging table created!")

Validation staging table created!


In [4]:
con.execute("""
CREATE OR REPLACE TABLE hotel_lake.silver.quarantined_bookings AS
SELECT *
FROM hotel_lake.silver.booking_validation
WHERE rejection_reason IS NOT NULL
""")

quarantine_count = con.execute("""
SELECT COUNT(*)
FROM hotel_lake.silver.quarantined_bookings
""").fetchone()[0]

print("Quarantined records:", quarantine_count)

display(con.execute("""
SELECT rejection_reason, COUNT(*) AS records
FROM hotel_lake.silver.quarantined_bookings
GROUP BY rejection_reason
ORDER BY records DESC
""").df())

Quarantined records: 181


,rejection_reason,records
0,No guests,180
1,Invalid ADR,1


In [5]:
con.execute("""
CREATE OR REPLACE TABLE hotel_lake.silver.clean_bookings AS
SELECT
    * EXCLUDE (Country, Children, rejection_reason),

    COALESCE(NULLIF(TRIM(Country), ''), 'Unknown')
        AS Country,

    COALESCE(Children, 0)
        AS Children,

    StaysInWeekendNights + StaysInWeekNights
        AS TotalNights,

    Adults + COALESCE(Children, 0) + Babies
        AS TotalGuests

FROM hotel_lake.silver.booking_validation
WHERE rejection_reason IS NULL
""")

silver_count = con.execute("""
SELECT COUNT(*)
FROM hotel_lake.silver.clean_bookings
""").fetchone()[0]

print("Clean Silver records:", silver_count)
print("Quarantined records:", quarantine_count)

assert silver_count + quarantine_count == 119390

print("Silver reconciliation passed!")

Clean Silver records: 119209
Quarantined records: 181
Silver reconciliation passed!


### Silver Data Quality Report

This report summarizes the number of source records, validated records, quarantined records, missing values, duplicate-looking records, and source-key uniqueness.

The dataset does not contain an original booking ID. Therefore, identical-looking observations are reported but not automatically removed.

In [6]:
# Silver Data Quality Report

bronze_count = con.execute("""
SELECT COUNT(*)
FROM hotel_lake.bronze.hotel_bookings
""").fetchone()[0]

silver_count = con.execute("""
SELECT COUNT(*)
FROM hotel_lake.silver.clean_bookings
""").fetchone()[0]

quarantine_count = con.execute("""
SELECT COUNT(*)
FROM hotel_lake.silver.quarantined_bookings
""").fetchone()[0]

# Count missing values in original Bronze data
missing_values = con.execute("""
SELECT
    SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END)
        AS missing_country,
    SUM(CASE WHEN Children IS NULL THEN 1 ELSE 0 END)
        AS missing_children
FROM hotel_lake.bronze.hotel_bookings
""").df()

# Count identical-looking booking rows based on original source columns.
# Exclude ingestion metadata: it describes the load, not the booking.
identical_booking_duplicates = con.execute("""
SELECT COALESCE(SUM(duplicate_group_size - 1), 0)
FROM (
    SELECT
        * EXCLUDE (source_name, ingested_at, load_id, source_row_number),
        COUNT(*) AS duplicate_group_size
    FROM hotel_lake.bronze.hotel_bookings
    GROUP BY ALL
    HAVING COUNT(*) > 1
) duplicate_groups
""").fetchone()[0]

# Independently check whether source identifiers repeat.
duplicate_source_identifiers = con.execute("""
SELECT COUNT(*) - COUNT(DISTINCT (source_name, load_id, source_row_number))
FROM hotel_lake.bronze.hotel_bookings
""").fetchone()[0]

# Check source-key uniqueness in Silver
unique_keys = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT source_name, load_id, source_row_number
    FROM hotel_lake.silver.clean_bookings
    GROUP BY source_name, load_id, source_row_number
)
""").fetchone()[0]

quality_report = pd.DataFrame({
    "Metric": [
        "Bronze records",
        "Clean Silver records",
        "Quarantined records",
        "Identical-looking booking rows (extra copies)",
        "Duplicate source identifiers",
        "Missing Country values in Bronze",
        "Missing Children values in Bronze",
        "Unique Silver source identifiers"
    ],
    "Count": [
    bronze_count,
    silver_count,
    quarantine_count,
    int(identical_booking_duplicates),
    int(duplicate_source_identifiers),
    int(missing_values["missing_country"].iloc[0]),
    int(missing_values["missing_children"].iloc[0]),
    unique_keys
]
})

display(quality_report)

assert bronze_count == silver_count + quarantine_count
assert unique_keys == silver_count

print("Silver quality checks passed!")

,Metric,Count
0,Bronze records,119390
1,Clean Silver records,119209
2,Quarantined records,181
3,Identical-looking booking rows (extra copies),31994
4,Duplicate source identifiers,0
5,Missing Country values in Bronze,488
6,Missing Children values in Bronze,4
7,Unique Silver source identifiers,119209


Silver quality checks passed!


In [7]:
quarantine_report = con.execute("""
SELECT
    rejection_reason,
    COUNT(*) AS record_count
FROM hotel_lake.silver.quarantined_bookings
GROUP BY rejection_reason
ORDER BY record_count DESC
""").df()

display(quarantine_report)

,rejection_reason,record_count
0,No guests,180
1,Invalid ADR,1
